
🎯 ENTRY LONG:
├─ Ichimoku: Tenkan > Kijun
├─ Ichimoku: Chikou > Close[26 bars trước]
├─ Ichimoku: Close > Kumo Cloud (trên mây)
└─ MACD: MACD line cắt LÊN Signal line

🎯 ENTRY SHORT:
├─ Ichimoku: Tenkan < Kijun
├─ Ichimoku: Chikou < Close[26 bars trước]
├─ Ichimoku: Close < Kumo Cloud (dưới mây)
└─ MACD: MACD line cắt XUỐNG Signal line

🛡️ RISK MANAGEMENT (tự bổ sung):
├─ SL: 2.0 * ATR(14)
├─ TP: 4.0 * ATR(14)
├─ R:R Ratio: 1:2
├─ Risk per trade: 1%
└─ Max hold: 50 bars

func (bt *Backtester) Run() {
	indices := bt.DF.Index

	for i := 0; i < len(bt.DF.Rows); i++ {
		currentBar := bt.DF.Rows[i]
		idx := indices[i]

		// ===== A. QUẢN LÝ LỆNH ĐANG MỞ =====
		if bt.Position != nil {
			barsHeld := i - bt.EntryBarIdx

			if barsHeld >= bt.MaxHoldBars {
				bt.ClosePosition(currentBar.Close, idx, "Time Exit")
				bt.AppendEquity(idx)
				continue
			}

			if *bt.Position == "LONG" {
				if currentBar.Low <= bt.SL {
					bt.ClosePosition(bt.SL, idx, "Stop Loss")
					bt.AppendEquity(idx)
					continue
				}

				if currentBar.High >= bt.TP {
					bt.ClosePosition(bt.TP, idx, "Take Profit")
					bt.AppendEquity(idx)
					continue
				}
			} else if *bt.Position == "SHORT" {
				if currentBar.High >= bt.SL {
					bt.ClosePosition(bt.SL, idx, "Stop Loss")
					bt.AppendEquity(idx)
					continue
				}

				if currentBar.Low <= bt.TP {
					bt.ClosePosition(bt.TP, idx, "Take Profit")
					bt.AppendEquity(idx)
					continue
				}
			}
		}

		// ===== B. TÍN HIỆU ENTRY =====
		if bt.Position == nil {
			if i < 26 {
				bt.AppendEquity(idx)
				continue
			}

			current := currentBar
			prev := bt.DF.Rows[i-1]
			close26Ago := bt.DF.Rows[i-26].Close

			// Ichimoku LONG
			ichimokuLong := current.TenkanSen > current.KijunSen &&
				current.ChikouSpan > close26Ago &&
				current.Close > current.CloudTop

			// Ichimoku SHORT
			ichimokuShort := current.TenkanSen < current.KijunSen &&
				current.ChikouSpan < close26Ago &&
				current.Close < current.CloudBottom

			// MACD crossover
			macdCrossUp := prev.MACD <= prev.MACDSignal &&
				current.MACD > current.MACDSignal

			macdCrossDown := prev.MACD >= prev.MACDSignal &&
				current.MACD < current.MACDSignal

			longSignal := ichimokuLong && macdCrossUp
			shortSignal := ichimokuShort && macdCrossDown

			if longSignal {
				atr := current.ATR
				entryPrice := current.Close
				slPrice := entryPrice - bt.SLATRMultiplier*atr
				tpPrice := entryPrice + bt.TPATRMultiplier*atr

				bt.OpenPosition(
					"LONG",
					entryPrice,
					idx,
					i,
					slPrice,
					tpPrice,
				)
			} else if shortSignal {
				atr := current.ATR
				entryPrice := current.Close
				slPrice := entryPrice + bt.SLATRMultiplier*atr
				tpPrice := entryPrice - bt.TPATRMultiplier*atr

				bt.OpenPosition(
					"SHORT",
					entryPrice,
					idx,
					i,
					slPrice,
					tpPrice,
				)
			}
		}

		// ===== C. APPEND EQUITY =====
		bt.AppendEquity(idx)
	}
}

In [6]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import time
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =========================================================
# 1. HÀM TẢI DỮ LIỆU
# =========================================================
def str_to_millis(date_str):
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    return int(dt.timestamp() * 1000)

def fetch_binance_kline_df(symbol, interval, start_str, end_str):
    base_url = "https://api.binance.com/"
    api_url = "api/v3/klines"
    general_url = base_url + api_url

    start_time = str_to_millis(start_str)
    end_dt = datetime.strptime(end_str, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
    end_time = int(end_dt.timestamp() * 1000)

    all_klines = []
    print(f"⏳ Đang tải dữ liệu {symbol} khung {interval}...")

    while start_time < end_time:
        params = {
            "symbol": symbol.upper(),
            "interval": interval,
            "startTime": start_time,
            "endTime": end_time,
            "limit": 1000
        }
        try:
            response = requests.get(general_url, params=params, timeout=10)
            if response.status_code != 200:
                print("Error:", response.status_code, response.text)
                break

            data = response.json()
            if not data:
                break

            all_klines.extend(data)
            last_ts = data[-1][0]
            start_time = last_ts + 1
            time.sleep(0.05)

        except Exception as e:
            print(f"Lỗi: {e}")
            break

    if not all_klines:
        return pd.DataFrame()

    df = pd.DataFrame(all_klines, columns=[
        "timestamp", "Open", "High", "Low", "Close", "Volume",
        "close_time", "quote_asset_volume", "num_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ])
    df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df.set_index("timestamp", inplace=True)
    df = df.astype(float)
    return df




In [7]:
# =========================================================
# 2. HÀM TÍNH INDICATORS
# =========================================================
def calculate_indicators(df):
    data = df.copy()

    # ===== ICHIMOKU =====
    high_9 = data["High"].rolling(window=9).max()
    low_9 = data["Low"].rolling(window=9).min()
    data["Tenkan_sen"] = (high_9 + low_9) / 2

    high_26 = data["High"].rolling(window=26).max()
    low_26 = data["Low"].rolling(window=26).min()
    data["Kijun_sen"] = (high_26 + low_26) / 2

    high_52 = data["High"].rolling(window=52).max()
    low_52 = data["Low"].rolling(window=52).min()

    # giữ đúng phong cách các template trước
    data["Senkou_A"] = ((data["Tenkan_sen"] + data["Kijun_sen"]) / 2).shift(26)
    data["Senkou_B"] = ((high_52 + low_52) / 2).shift(26)
    data["Chikou_span"] = data["Close"].shift(26)

    data["Cloud_top"] = data[["Senkou_A", "Senkou_B"]].max(axis=1)
    data["Cloud_bottom"] = data[["Senkou_A", "Senkou_B"]].min(axis=1)

    # ===== MACD =====
    ema_12 = data["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = data["Close"].ewm(span=26, adjust=False).mean()
    data["MACD"] = ema_12 - ema_26
    data["MACD_signal"] = data["MACD"].ewm(span=9, adjust=False).mean()
    data["MACD_hist"] = data["MACD"] - data["MACD_signal"]

    # ===== ATR(14) =====
    period = 14
    data["prev_close"] = data["Close"].shift(1)
    data["h_l"] = data["High"] - data["Low"]
    data["h_pc"] = (data["High"] - data["prev_close"]).abs()
    data["l_pc"] = (data["Low"] - data["prev_close"]).abs()
    data["TR"] = data[["h_l", "h_pc", "l_pc"]].max(axis=1)
    data["ATR"] = data["TR"].ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    data.dropna(inplace=True)
    return data




In [8]:
# =========================================================
# 3. BACKTESTER
# =========================================================
class IchimokuMACDBacktester:
    def __init__(self, df, initial_balance=10000, fee_rate=0.001):
        self.df = df
        self.initial_balance = initial_balance
        self.balance = initial_balance
        self.fee_rate = fee_rate

        self.equity_curve = []
        self.trades = []

        self.position = None
        self.entry_price = 0.0
        self.entry_time = None
        self.entry_bar_idx = 0
        self.qty = 0.0

        self.sl = 0.0
        self.tp = 0.0
        self.entry_fee = 0.0

        # Risk management
        self.risk_pct = 0.01
        self.sl_atr_multiplier = 2.0
        self.tp_atr_multiplier = 4.0
        self.max_hold_bars = 50

    def run(self):
        indices = self.df.index

        for i in range(len(self.df)):
            current_bar = self.df.iloc[i]
            idx = indices[i]

            # ===== A. QUẢN LÝ LỆNH ĐANG MỞ =====
            if self.position is not None:
                bars_held = i - self.entry_bar_idx

                if bars_held >= self.max_hold_bars:
                    self.close_position(current_bar["Close"], idx, "Time Exit")
                    self.append_equity(idx)
                    continue

                if self.position == "LONG":
                    if current_bar["Low"] <= self.sl:
                        self.close_position(self.sl, idx, "Stop Loss")
                        self.append_equity(idx)
                        continue

                    if current_bar["High"] >= self.tp:
                        self.close_position(self.tp, idx, "Take Profit")
                        self.append_equity(idx)
                        continue

                elif self.position == "SHORT":
                    if current_bar["High"] >= self.sl:
                        self.close_position(self.sl, idx, "Stop Loss")
                        self.append_equity(idx)
                        continue

                    if current_bar["Low"] <= self.tp:
                        self.close_position(self.tp, idx, "Take Profit")
                        self.append_equity(idx)
                        continue

            # ===== B. TÍN HIỆU ENTRY =====
            if self.position is None:
                if i < 26:
                    self.append_equity(idx)
                    continue

                current = current_bar
                prev = self.df.iloc[i - 1]
                close_26_ago = self.df.iloc[i - 26]["Close"]

                # Ichimoku LONG
                ichimoku_long = (
                    current["Tenkan_sen"] > current["Kijun_sen"] and
                    current["Chikou_span"] > close_26_ago and
                    current["Close"] > current["Cloud_top"]
                )

                # Ichimoku SHORT
                ichimoku_short = (
                    current["Tenkan_sen"] < current["Kijun_sen"] and
                    current["Chikou_span"] < close_26_ago and
                    current["Close"] < current["Cloud_bottom"]
                )

                # MACD crossover
                macd_cross_up = (
                    prev["MACD"] <= prev["MACD_signal"] and
                    current["MACD"] > current["MACD_signal"]
                )

                macd_cross_down = (
                    prev["MACD"] >= prev["MACD_signal"] and
                    current["MACD"] < current["MACD_signal"]
                )

                long_signal = ichimoku_long and macd_cross_up
                short_signal = ichimoku_short and macd_cross_down

                if long_signal:
                    atr = current["ATR"]
                    entry_price = current["Close"]
                    sl_price = entry_price - self.sl_atr_multiplier * atr
                    tp_price = entry_price + self.tp_atr_multiplier * atr
                    self.open_position(
                        pos_type="LONG",
                        price=entry_price,
                        time_idx=idx,
                        bar_idx=i,
                        sl_price=sl_price,
                        tp_price=tp_price
                    )

                elif short_signal:
                    atr = current["ATR"]
                    entry_price = current["Close"]
                    sl_price = entry_price + self.sl_atr_multiplier * atr
                    tp_price = entry_price - self.tp_atr_multiplier * atr
                    self.open_position(
                        pos_type="SHORT",
                        price=entry_price,
                        time_idx=idx,
                        bar_idx=i,
                        sl_price=sl_price,
                        tp_price=tp_price
                    )

            # ===== C. APPEND EQUITY =====
            self.append_equity(idx)

    def append_equity(self, idx):
        mark_to_market = self.balance

        if self.position == "LONG":
            current_price = self.df.loc[idx, "Close"]
            unrealized_gross = (current_price - self.entry_price) * self.qty
            est_exit_fee = current_price * self.qty * self.fee_rate
            mark_to_market += unrealized_gross - est_exit_fee

        elif self.position == "SHORT":
            current_price = self.df.loc[idx, "Close"]
            unrealized_gross = (self.entry_price - current_price) * self.qty
            est_exit_fee = current_price * self.qty * self.fee_rate
            mark_to_market += unrealized_gross - est_exit_fee

        self.equity_curve.append({
            "time": idx,
            "equity": mark_to_market
        })

    def open_position(self, pos_type, price, time_idx, bar_idx, sl_price, tp_price):
        self.position = pos_type
        self.entry_price = price
        self.entry_time = time_idx
        self.entry_bar_idx = bar_idx
        self.sl = sl_price
        self.tp = tp_price

        risk_amount = self.balance * self.risk_pct
        risk_per_unit = abs(price - self.sl) + self.fee_rate * (price + self.sl)
        self.qty = risk_amount / risk_per_unit if risk_per_unit > 0 else 0

        self.entry_fee = price * self.qty * self.fee_rate
        self.balance -= self.entry_fee

    def close_position(self, price, time_idx, reason):
        exit_fee = price * self.qty * self.fee_rate

        if self.position == "LONG":
            pnl_gross = (price - self.entry_price) * self.qty
        elif self.position == "SHORT":
            pnl_gross = (self.entry_price - price) * self.qty
        else:
            pnl_gross = 0.0

        pnl_net_to_balance = pnl_gross - exit_fee
        self.balance += pnl_net_to_balance

        self.trades.append({
            "Entry Time": self.entry_time,
            "Exit Time": time_idx,
            "Type": self.position,
            "Entry": self.entry_price,
            "Exit": price,
            "Qty": self.qty,
            "Gross PnL": pnl_gross,
            "Entry Fee": self.entry_fee,
            "Exit Fee": exit_fee,
            "Net PnL": pnl_gross - self.entry_fee - exit_fee,
            "Reason": reason,
            "Balance": self.balance
        })

        self.position = None
        self.entry_price = 0.0
        self.entry_time = None
        self.entry_bar_idx = 0
        self.qty = 0.0
        self.sl = 0.0
        self.tp = 0.0
        self.entry_fee = 0.0




In [9]:
# =========================================================
# 4. METRICS
# =========================================================
def calculate_advanced_metrics(trades_df, equity_data, initial_balance):
    eq_df = pd.DataFrame(equity_data)
    eq_df.set_index("time", inplace=True)
    eq_df = eq_df[~eq_df.index.duplicated(keep="last")]

    final_equity = eq_df["equity"].iloc[-1]
    total_return = (final_equity - initial_balance) / initial_balance

    days = (eq_df.index[-1] - eq_df.index[0]).days
    years = days / 365.25
    cagr = ((final_equity / initial_balance) ** (1 / years)) - 1 if years > 0 else 0

    running_max = eq_df["equity"].cummax()
    drawdown = (eq_df["equity"] - running_max) / running_max
    max_dd = drawdown.min()

    eq_df["returns"] = eq_df["equity"].pct_change()
    mean_ret = eq_df["returns"].mean()
    std_ret = eq_df["returns"].std()
    bars_per_year = len(eq_df) / years if years > 0 else 8760
    sharpe = (mean_ret / std_ret) * np.sqrt(bars_per_year) if std_ret != 0 else 0

    total_trades = len(trades_df)
    win_trades = len(trades_df[trades_df["Net PnL"] > 0])
    win_rate = win_trades / total_trades if total_trades > 0 else 0
    avg_pnl = trades_df["Net PnL"].mean() if total_trades > 0 else 0

    gross_profit = trades_df[trades_df["Net PnL"] > 0]["Net PnL"].sum()
    gross_loss = abs(trades_df[trades_df["Net PnL"] < 0]["Net PnL"].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else 0

    total_entry_fees = trades_df["Entry Fee"].sum() if total_trades > 0 else 0
    total_exit_fees = trades_df["Exit Fee"].sum() if total_trades > 0 else 0
    total_fees = total_entry_fees + total_exit_fees

    return {
        "Final Equity": final_equity,
        "Total Return": total_return,
        "CAGR (approx)": cagr,
        "Max Drawdown": max_dd,
        "Sharpe": sharpe,
        "Trades": total_trades,
        "Winrate": win_rate,
        "Avg PnL/Trade": avg_pnl,
        "Profit Factor": profit_factor,
        "Total Entry Fees": total_entry_fees,
        "Total Exit Fees": total_exit_fees,
        "Total Fees": total_fees,
        "equity_df": eq_df
    }

def print_report_and_plot(metrics, symbol):
    print("\n" + "=" * 70)
    print("📊 ICHIMOKU + MACD TREND FOLLOWING STRATEGY (WITH FEES)")
    print("=" * 70)
    print(f"     Final Equity: ${metrics['Final Equity']:,.2f}")
    print(f"     Total Return: {metrics['Total Return']*100:.2f}%")
    print(f"    CAGR (approx): {metrics['CAGR (approx)']*100:.2f}%")
    print(f"     Max Drawdown: {metrics['Max Drawdown']*100:.2f}%")
    print(f"           Sharpe: {metrics['Sharpe']:.4f}")
    print(f"           Trades: {metrics['Trades']}")
    print(f"          Winrate: {metrics['Winrate']*100:.2f}%")
    print(f" Avg Net PnL/Trade: ${metrics['Avg PnL/Trade']:.2f}")
    print(f"    Profit Factor: {metrics['Profit Factor']:.4f}")
    print(f" Total Entry Fees: ${metrics['Total Entry Fees']:.2f}")
    print(f"  Total Exit Fees: ${metrics['Total Exit Fees']:.2f}")
    print(f"      Total Fees : ${metrics['Total Fees']:.2f}")
    print("=" * 70)

    eq_df = metrics["equity_df"]
    plt.figure(figsize=(14, 7))
    plt.plot(eq_df.index, eq_df["equity"], linewidth=2, color="#E63946", label="Equity")
    plt.fill_between(eq_df.index, eq_df["equity"], alpha=0.3, color="#E63946")
    plt.axhline(y=10000, color="gray", linestyle="--", linewidth=1, alpha=0.5, label="Initial Balance")

    plt.title(f"📈 Equity Curve - {symbol} (Ichimoku + MACD)", fontsize=16, fontweight="bold")
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Equity ($)", fontsize=12)
    plt.legend(loc="best")
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()




In [10]:
# =========================================================
# 5. MAIN
# =========================================================
if __name__ == "__main__":
    symbols = ["ETHUSDT", "BTCUSDT", "BNBUSDT", "SOLUSDT","XRPUSDT"]
    intervals = ["15m","30m","1h", "2h", "4h", "1d"]

    start_date = "01/01/2024"
    end_date = "18/06/2026"
    initial_balance = 10000
    fee_rate = 0.001  # 0.1% mỗi chiều

    all_results = []

    for symbol in symbols:
        for interval in intervals:
            print("\n" + "=" * 80)
            print(f"🚀 Đang test: {symbol} | Timeframe: {interval}")
            print("=" * 80)

            try:
                df_raw = fetch_binance_kline_df(symbol, interval, start_date, end_date)

                if len(df_raw) <= 200:
                    print(f"❌ Không đủ dữ liệu cho {symbol} - {interval}")
                    continue

                print(f"✅ Tải được {len(df_raw)} nến. Đang tính indicators...")
                df_calc = calculate_indicators(df_raw)
                print(f"✅ Indicators sẵn sàng. Dữ liệu sau filter: {len(df_calc)} nến")

                print("🔄 Đang chạy backtest...")
                bt = IchimokuMACDBacktester(
                    df=df_calc,
                    initial_balance=initial_balance,
                    fee_rate=fee_rate
                )
                bt.run()

                trades_df = pd.DataFrame(bt.trades)

                if not trades_df.empty:
                    metrics = calculate_advanced_metrics(
                        trades_df,
                        bt.equity_curve,
                        bt.initial_balance
                    )

                    print_report_and_plot(metrics, f"{symbol} ({interval})")

                    # Lưu kết quả tóm tắt
                    all_results.append({
                        "symbol": symbol,
                        "interval": interval,
                        "trades": len(trades_df),
                        "Equity": metrics.get("Final Equity", None),
                        "cagr": metrics.get("CAGR (approx)", None),
                        "max_drawdown": metrics.get("Max Drawdown", None),
                        "sharpe": metrics.get("Sharpe", None),
                        "win_rate_pct": metrics.get("Winrate", None),
                    })
                else:
                    print("⚠️ Không có lệnh nào được thực hiện.")

            except Exception as e:
                print(f"❌ Lỗi khi test {symbol} - {interval}: {e}")

    # =========================================================
    # Tổng hợp kết quả
    # =========================================================
    if all_results:
        summary_df = pd.DataFrame(all_results)
        summary_df = summary_df.sort_values(
            by="Equity",
            ascending=False,
            na_position="last"
        )

        print("\n" + "=" * 100)
        print("📊 TỔNG HỢP KẾT QUẢ BACKTEST")
        print("=" * 100)
        print(summary_df)

    else:
        print("\n⚠️ Không có kết quả nào để tổng hợp.")


🚀 Đang test: ETHUSDT | Timeframe: 15m
⏳ Đang tải dữ liệu ETHUSDT khung 15m...
✅ Tải được 86400 nến. Đang tính indicators...
✅ Indicators sẵn sàng. Dữ liệu sau filter: 86323 nến
🔄 Đang chạy backtest...
⚠️ Không có lệnh nào được thực hiện.

🚀 Đang test: ETHUSDT | Timeframe: 30m
⏳ Đang tải dữ liệu ETHUSDT khung 30m...
✅ Tải được 43200 nến. Đang tính indicators...
✅ Indicators sẵn sàng. Dữ liệu sau filter: 43123 nến
🔄 Đang chạy backtest...
⚠️ Không có lệnh nào được thực hiện.

🚀 Đang test: ETHUSDT | Timeframe: 1h
⏳ Đang tải dữ liệu ETHUSDT khung 1h...
✅ Tải được 21600 nến. Đang tính indicators...
✅ Indicators sẵn sàng. Dữ liệu sau filter: 21523 nến
🔄 Đang chạy backtest...
⚠️ Không có lệnh nào được thực hiện.

🚀 Đang test: ETHUSDT | Timeframe: 2h
⏳ Đang tải dữ liệu ETHUSDT khung 2h...
✅ Tải được 10800 nến. Đang tính indicators...
✅ Indicators sẵn sàng. Dữ liệu sau filter: 10723 nến
🔄 Đang chạy backtest...
⚠️ Không có lệnh nào được thực hiện.

🚀 Đang test: ETHUSDT | Timeframe: 4h
⏳ Đang tải